# Progetto Finale: Classificazione Binaria di Fiori

## Obiettivo del Progetto

Questo notebook implementa un sistema di classificazione binaria per distinguere tra due tipi di fiori:
- **daisy** (margherita)
- **dandelion** (dente di leone)

### Perché Transfer Learning?
Utilizziamo un modello pre-addestrato su ImageNet (ResNet18) perché:
- I dataset di fiori sono spesso piccoli e non sufficienti per addestrare da zero una rete neurale profonda
- Le feature apprese su ImageNet (forme, texture, pattern) sono trasferibili a molti task di classificazione
- Riduce drasticamente il tempo di training e migliora le performance

### Perché Data Augmentation?
L'aumento dei dati è fondamentale quando si lavora con dataset limitati:
- Aumenta la varietà dei dati senza raccogliere nuove immagini
- Aiuta il modello a generalizzare meglio evitando overfitting
- Simula variazioni reali (illuminazione, rotazioni, zoom) che il modello incontrerà in produzione

### Perché Macro F1-Score?
Il macro F1-score è la metrica principale perché:
- È robusto a dataset sbilanciati (calcola F1 per classe e fa la media)
- Bilancia precision e recall senza privilegiare classi più frequenti
- È più informativo della semplice accuracy quando le classi non sono perfettamente bilanciate

## 1. Setup Ambiente

Installiamo le dipendenze necessarie e configuriamo l'ambiente di lavoro.

In [ ]:
# Installa timm se non presente
try:
    import timm
    print(f"timm già installato: versione {timm.__version__}")
except ImportError:
    print("Installazione timm...")
    !pip install timm -q
    import timm
    print("timm installato con successo!")

In [ ]:
# Import delle librerie necessarie
import os
import shutil
import tarfile
from pathlib import Path
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import functional as F

import timm

print("Tutte le librerie importate con successo!")

In [ ]:
# Configurazione device e seed per riproducibilità
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA versione: {torch.version.cuda}")

def set_seed(seed=42):
    """Imposta il seed per riproducibilità completa"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Per performance riproducibili (può rallentare leggermente)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print(f"PyTorch versione: {torch.__version__}")
print("Seed impostato a 42 per riproducibilità")

## 2. Download e Estrazione Dataset

Scarichiamo il dataset e verifichiamo la struttura delle cartelle. Il codice è robusto e trova automaticamente i path corretti.

In [ ]:
# URL del dataset
dataset_url = "https://proai-datasets.s3.eu-west-3.amazonaws.com/progetto-finale-flowes.tar.gz"
dataset_filename = "progetto-finale-flowes.tar.gz"
extracted_dir = "progetto-finale-flowes"

# Download del dataset
if not os.path.exists(dataset_filename):
    print("Download del dataset in corso...")
    !wget -q {dataset_url} -O {dataset_filename}
    print("Download completato!")
else:
    print("File dataset già presente.")

# Estrazione
if os.path.exists(extracted_dir):
    print(f"Directory {extracted_dir} già esistente, skip estrazione.")
else:
    print("Estrazione del dataset...")
    with tarfile.open(dataset_filename, "r:gz") as tar:
        tar.extractall()
    print("Estrazione completata!")

print(f"\nContenuto directory corrente:")
!ls -la

In [ ]:
# Funzione per ispezionare la struttura delle directory
def inspect_directory(root_dir, max_depth=3, current_depth=0):
    """Stampa la struttura delle directory fino a max_depth"""
    if current_depth >= max_depth:
        return
    
    root_path = Path(root_dir)
    if not root_path.exists():
        print(f"Directory {root_dir} non trovata!")
        return
    
    indent = "  " * current_depth
    print(f"{indent}{root_path.name}/")
    
    # Conta immagini nelle directory
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    image_count = sum(1 for f in root_path.rglob('*') if f.suffix.lower() in image_extensions)
    
    if current_depth < max_depth - 1:
        try:
            items = sorted(root_path.iterdir())
            for item in items:
                if item.is_dir():
                    inspect_directory(item, max_depth, current_depth + 1)
                elif item.suffix.lower() in image_extensions and current_depth == max_depth - 1:
                    print(f"{indent}  {item.name}")
        except PermissionError:
            pass
    
    if current_depth == 0:
        print(f"\nTotale immagini trovate: {image_count}")

print("Struttura directory dataset:")
inspect_directory(extracted_dir, max_depth=3)

In [ ]:
# Funzione robusta per trovare automaticamente i path di train/val/test
def find_dataset_paths(root_dir):
    """
    Trova automaticamente i path di train, val (o valid), test.
    Cerca ricorsivamente se necessario.
    """
    root_path = Path(root_dir)
    expected_splits = ['train', 'val', 'valid', 'test']
    found_paths = {}
    
    # Prima cerca direttamente nella root
    for split in expected_splits:
        potential_path = root_path / split
        if potential_path.exists() and potential_path.is_dir():
            # Verifica che contenga sottocartelle di classe
            subdirs = [d for d in potential_path.iterdir() if d.is_dir()]
            if len(subdirs) >= 2:  # Almeno 2 classi
                found_paths[split] = potential_path
                print(f"✓ Trovato {split}: {potential_path}")
    
    # Se non trovato, cerca ricorsivamente
    if 'train' not in found_paths or 'test' not in found_paths:
        print("\nCerca ricorsivamente...")
        for split in expected_splits:
            if split in found_paths:
                continue
            # Cerca in tutte le sottodirectory
            for potential_dir in root_path.rglob(split):
                if potential_dir.is_dir():
                    subdirs = [d for d in potential_dir.iterdir() if d.is_dir()]
                    if len(subdirs) >= 2:
                        found_paths[split] = potential_dir
                        print(f"✓ Trovato {split} (ricorsivo): {potential_dir}")
                        break
    
    # Normalizza 'valid' -> 'val'
    if 'valid' in found_paths and 'val' not in found_paths:
        found_paths['val'] = found_paths.pop('valid')
    
    return found_paths

# Trova i path
dataset_paths = find_dataset_paths(extracted_dir)

if not dataset_paths:
    raise ValueError("Nessun path di dataset trovato! Verifica la struttura delle cartelle.")

In [ ]:
# Verifica e stampa riepilogo per split e classe
def count_images_by_class(dataset_path):
    """Conta immagini per classe in un dataset"""
    counts = defaultdict(int)
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    
    for class_dir in dataset_path.iterdir():
        if class_dir.is_dir():
            class_name = class_dir.name
            count = sum(1 for f in class_dir.rglob('*') 
                       if f.suffix.lower() in image_extensions)
            counts[class_name] = count
    
    return counts

print("=" * 60)
print("RIEPILOGO DATASET")
print("=" * 60)

for split_name, split_path in dataset_paths.items():
    print(f"\n{split_name.upper()}:")
    class_counts = count_images_by_class(split_path)
    total = sum(class_counts.values())
    print(f"  Totale immagini: {total}")
    for class_name, count in sorted(class_counts.items()):
        print(f"    - {class_name}: {count} immagini")

## 3. DataLoader con Data Augmentation

Creiamo i DataLoader con trasformazioni appropriate:
- **Training**: augmentation aggressiva per aumentare la varietà dei dati
- **Validation/Test**: solo resize e crop per valutazione corretta

In [ ]:
# Parametri
img_size = 224
batch_size = 32

# Statistiche ImageNet per normalizzazione (standard per modelli pre-addestrati)
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Transform per training: augmentation aggressiva
# Perché queste augmentation?
# - RandomResizedCrop: simula zoom e crop casuali (variazione scala)
# - RandomHorizontalFlip: simmetria orizzontale (molto comune in natura)
# - ColorJitter: varia illuminazione e colore (robustezza a condizioni diverse)
# - RandomRotation: piccole rotazioni (fiori possono essere fotografati da angoli diversi)
train_transform = transforms.Compose([
    transforms.Resize(256),  # Resize più grande per poi fare crop
    transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),  # Crop casuale con zoom
    transforms.RandomHorizontalFlip(p=0.5),  # Flip orizzontale
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Variazione colore
    transforms.RandomRotation(degrees=15),  # Rotazione piccola
    transforms.ToTensor(),  # Converti in tensor [0, 1]
    transforms.Normalize(mean=mean, std=std)  # Normalizza con ImageNet stats
])

# Transform per validation/test: solo preprocessing base
# Niente augmentation per valutare correttamente le performance
val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(img_size),  # Crop centrale (deterministico)
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

print("Transform create con successo!")

In [ ]:
# Crea i dataset usando ImageFolder (richiede struttura: split/class_name/images)
train_dataset = datasets.ImageFolder(
    root=str(dataset_paths['train']),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=str(dataset_paths['val']),
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    root=str(dataset_paths['test']),
    transform=val_test_transform
)

# Verifica che le classi siano consistenti
assert train_dataset.classes == val_dataset.classes == test_dataset.classes, \
    "Le classi devono essere le stesse in tutti gli split!"

print(f"Classi trovate: {train_dataset.classes}")
print(f"Class to index: {train_dataset.class_to_idx}")
print(f"\nTrain: {len(train_dataset)} immagini")
print(f"Val: {len(val_dataset)} immagini")
print(f"Test: {len(test_dataset)} immagini")

In [ ]:
# Crea i DataLoader
# num_workers: su Colab usa 2 per evitare problemi, su locale puoi aumentare
num_workers = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # Shuffle solo per training
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,  # No shuffle per validation
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

# Verifica shape di un batch
sample_batch = next(iter(train_loader))
images, labels = sample_batch
print(f"\nShape batch training:")
print(f"  Images: {images.shape}")  # [batch_size, 3, 224, 224]
print(f"  Labels: {labels.shape}")  # [batch_size]
print(f"  Label range: {labels.min().item()} - {labels.max().item()}")

## 4. Modello: ResNet18 con Transfer Learning

Carichiamo ResNet18 pre-addestrato su ImageNet e adattiamolo al nostro task binario.
Strategia a due fasi:
1. **Fase 1**: Freeze del backbone, alleniamo solo la testa (classificatore)
2. **Fase 2**: Unfreeze e fine-tuning con learning rate più basso

In [ ]:
# Crea modello ResNet18 pre-addestrato
# timm crea automaticamente il modello con num_classes=2
model = timm.create_model("resnet18", pretrained=True, num_classes=2)
model = model.to(device)

print(f"Modello ResNet18 creato e spostato su {device}")
print(f"Numero totale parametri: {sum(p.numel() for p in model.parameters()):,}")

# Funzione per freeze/unfreeze il backbone
def freeze_backbone(model):
    """Freeze tutti i layer tranne il classificatore finale"""
    for name, param in model.named_parameters():
        if 'fc' not in name:  # 'fc' è il classificatore finale in ResNet
            param.requires_grad = False
        else:
            param.requires_grad = True
    print("Backbone congelato, solo classificatore trainable")

def unfreeze_backbone(model):
    """Rendi tutti i parametri trainable"""
    for param in model.parameters():
        param.requires_grad = True
    print("Tutti i parametri sono ora trainable")

# Conta parametri trainable
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParametri trainable prima del freeze: {count_trainable_params(model):,}")

# Fase 1: freeze backbone
freeze_backbone(model)
print(f"Parametri trainable dopo freeze: {count_trainable_params(model):,}")

In [ ]:
# Loss e Optimizer
# CrossEntropyLoss è standard per classificazione multi-classe (include softmax)
criterion = nn.CrossEntropyLoss()

# AdamW: variante di Adam con weight decay corretto (separato dal momentum)
# Learning rate più alto per la fase 1 (solo classificatore)
lr_phase1 = 3e-4
weight_decay = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=lr_phase1,
    weight_decay=weight_decay
)

# Scheduler: StepLR riduce il learning rate ogni N epoch
# Perché usare uno scheduler?
# - All'inizio servono step grandi per esplorare lo spazio
# - Verso la fine step piccoli per convergenza fine
# - Migliora la stabilità e le performance finali
step_size = 3  # Riduce LR ogni 3 epoch
gamma = 0.5  # Moltiplica LR per 0.5
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

print(f"Optimizer: AdamW (lr={lr_phase1}, weight_decay={weight_decay})")
print(f"Scheduler: StepLR (step_size={step_size}, gamma={gamma})")
print("Setup completato!")

## 5. Training Industrializzato

Implementiamo training con:
- **Checkpointing**: salva best model e ultimi 5 checkpoint
- **Early Stopping**: ferma se non migliora per patience epoch
- **Metriche**: loss, accuracy, macro F1-score

In [ ]:
# Directory per i checkpoint
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Addestra il modello per un'epoch"""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Passaggio in avanti
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Passaggio all'indietro
        loss.backward()
        optimizer.step()
        
        # Metriche
        running_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    # Calcola metriche
    epoch_loss = running_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss, accuracy, f1_macro

def evaluate(model, loader, criterion, device):
    """Valuta il modello su un dataset"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss, accuracy, f1_macro

print("Funzioni di training e evaluation create!")

In [ ]:
# Funzione per salvare checkpoint con rotazione
def save_checkpoint(model, optimizer, scheduler, epoch, metric_value, is_best=False, checkpoint_dir=checkpoint_dir):
    """Salva checkpoint con rotazione degli ultimi 5"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_metric': metric_value
    }
    
    # Salva best model
    if is_best:
        best_path = checkpoint_dir / "best.pth"
        torch.save(checkpoint, best_path)
        print(f"  ✓ Best model salvato (F1={metric_value:.4f})")
    
    # Salva last checkpoint con rotazione
    last_path = checkpoint_dir / "last_1.pth"
    
    # Ruota i checkpoint esistenti (last_1 -> last_2, last_2 -> last_3, etc.)
    for i in range(4, 0, -1):
        old_path = checkpoint_dir / f"last_{i}.pth"
        new_path = checkpoint_dir / f"last_{i+1}.pth"
        if old_path.exists():
            old_path.rename(new_path)
    
    # Elimina last_5 se esiste (dopo rotazione diventa last_6, ma non lo salviamo)
    last_5_path = checkpoint_dir / "last_5.pth"
    if last_5_path.exists():
        last_5_path.unlink()
    
    torch.save(checkpoint, last_path)
    print(f"  ✓ Checkpoint epoch {epoch} salvato")

def load_checkpoint(model, optimizer, scheduler, checkpoint_path):
    """Carica checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    return checkpoint['epoch'], checkpoint['best_metric']

print("Funzioni di checkpoint create!")

In [ ]:
# Training Fase 1: Freeze backbone, allena solo classificatore
print("=" * 60)
print("FASE 1: Training con backbone congelato")
print("=" * 60)

epochs_phase1 = 3
best_f1 = 0.0
patience = 4
patience_counter = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'train_f1': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': []
}

for epoch in range(1, epochs_phase1 + 1):
    print(f"\nEpoch {epoch}/{epochs_phase1}")
    print(f"Learning rate: {scheduler.get_last_lr()[0]:.6f}")
    
    # Training
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validation
    val_loss, val_acc, val_f1 = evaluate(
        model, val_loader, criterion, device
    )
    
    # Aggiorna scheduler
    scheduler.step()
    
    # Salva metriche
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # Checkpoint e early stopping
    is_best = val_f1 > best_f1
    if is_best:
        best_f1 = val_f1
        patience_counter = 0
        save_checkpoint(model, optimizer, scheduler, epoch, best_f1, is_best=True)
    else:
        patience_counter += 1
        save_checkpoint(model, optimizer, scheduler, epoch, best_f1, is_best=False)
    
    print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n⚠ Early stopping attivato! Nessun miglioramento per {patience} epoch.")
        break

print(f"\nFase 1 completata. Best F1: {best_f1:.4f}")

In [ ]:
# Fase 2: Unfreeze e fine-tuning
print("\n" + "=" * 60)
print("FASE 2: Fine-tuning completo (unfreeze backbone)")
print("=" * 60)

# Unfreeze tutto
unfreeze_backbone(model)
print(f"Parametri trainable dopo unfreeze: {count_trainable_params(model):,}")

# Learning rate più basso per fine-tuning (evita di distruggere feature pre-addestrate)
lr_phase2 = 1e-4
optimizer = optim.AdamW(model.parameters(), lr=lr_phase2, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

# Reset early stopping
patience_counter = 0
start_epoch = epochs_phase1 + 1
max_epochs = 10  # Massimo totale di epoch

for epoch in range(start_epoch, max_epochs + 1):
    print(f"\nEpoch {epoch}/{max_epochs}")
    print(f"Learning rate: {scheduler.get_last_lr()[0]:.6f}")
    
    # Training
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validation
    val_loss, val_acc, val_f1 = evaluate(
        model, val_loader, criterion, device
    )
    
    # Update scheduler
    scheduler.step()
    
    # Salva metriche
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # Checkpoint e early stopping
    is_best = val_f1 > best_f1
    if is_best:
        best_f1 = val_f1
        patience_counter = 0
        save_checkpoint(model, optimizer, scheduler, epoch, best_f1, is_best=True)
    else:
        patience_counter += 1
        save_checkpoint(model, optimizer, scheduler, epoch, best_f1, is_best=False)
    
    print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n⚠ Early stopping attivato! Nessun miglioramento per {patience} epoch.")
        break

print(f"\nTraining completato! Best F1 validation: {best_f1:.4f}")

## 6. Visualizzazione Training

Visualizziamo l'andamento delle metriche durante il training.

In [ ]:
# Plot delle metriche di training
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs_range, history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training e Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(epochs_range, history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training e Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Macro F1
axes[2].plot(epochs_range, history['train_f1'], label='Train F1', marker='o')
axes[2].plot(epochs_range, history['val_f1'], label='Val F1', marker='s')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro F1-Score')
axes[2].set_title('Training e Validation Macro F1-Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Valutazione su Test Set

Carichiamo il best model e valutiamo le performance sul test set con tutte le metriche richieste.

In [ ]:
# Carica best model
best_checkpoint_path = checkpoint_dir / "best.pth"
if best_checkpoint_path.exists():
    print(f"Caricamento best model da {best_checkpoint_path}")
    model.load_state_dict(torch.load(best_checkpoint_path, map_location=device)['model_state_dict'])
    print("Best model caricato!")
else:
    print("⚠ Best checkpoint non trovato, uso modello corrente")

# Valutazione su test set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Calcola metriche
test_accuracy = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, average='macro')
test_recall = recall_score(all_labels, all_preds, average='macro')
test_f1_macro = f1_score(all_labels, all_preds, average='macro')

print("=" * 60)
print("RISULTATI TEST SET")
print("=" * 60)
print(f"Accuracy:     {test_accuracy:.4f}")
print(f"Precision:    {test_precision:.4f} (macro)")
print(f"Recall:       {test_recall:.4f} (macro)")
print(f"F1-Score:     {test_f1_macro:.4f} (macro)")
print("=" * 60)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
class_names = train_dataset.classes

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

# Aggiungi testo nelle celle
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names, yticklabels=class_names,
       title='Confusion Matrix - Test Set',
       ylabel='True Label',
       xlabel='Predicted Label')

plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

## 8. Inference Demo

Mostriamo alcuni esempi di predizione su immagini del test set.

In [ ]:
def predict_image(model, image_path, transform, class_names, device):
    """Predice la classe di un'immagine"""
    # Carica e trasforma immagine
    from PIL import Image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predizione
    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim=1)
        pred_class_idx = output.argmax(dim=1).item()
        pred_class = class_names[pred_class_idx]
        confidence = probs[0][pred_class_idx].item()
    
    return pred_class, confidence, probs[0].cpu().numpy()

# Mostra 6 immagini random dal test set
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Seleziona 6 indici random
random_indices = np.random.choice(len(test_dataset), size=6, replace=False)

for idx, ax in zip(random_indices, axes):
    # Carica immagine originale (senza normalizzazione per visualizzazione)
    image_path, true_label_idx = test_dataset.samples[idx]
    true_label = class_names[true_label_idx]
    
    # Predizione
    pred_label, confidence, all_probs = predict_image(
        model, image_path, val_test_transform, class_names, device
    )
    
    # Carica immagine per visualizzazione
    from PIL import Image
    img = Image.open(image_path).convert('RGB')
    
    ax.imshow(img)
    ax.axis('off')
    
    # Colore: verde se corretto, rosso se sbagliato
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence:.2f})",
        color=color,
        fontsize=10
    )

plt.tight_layout()
plt.show()

## 9. Explainability: GradCAM

Implementiamo GradCAM per capire "dove guarda" la rete quando fa una predizione.
GradCAM combina:
- **Attivazioni** del layer convoluzionale (cosa ha "visto" la rete)
- **Gradienti** rispetto alla classe predetta (quanto è importante ogni regione)

Il risultato è una heatmap che evidenzia le regioni più importanti per la decisione.

In [ ]:
class GradCAM:
    """Implementazione semplice di GradCAM usando PyTorch hooks"""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Registra hooks
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        """Salva le attivazioni durante forward pass"""
        self.activations = output
    
    def save_gradient(self, module, grad_input, grad_output):
        """Salva i gradienti durante backward pass"""
        self.gradients = grad_output[0]
    
    def generate_cam(self, input_image, class_idx=None):
        """Genera la heatmap GradCAM"""
        # Reset gradienti e attivazioni
        self.gradients = None
        self.activations = None
        
        # Eval mode ma con gradienti abilitati
        self.model.eval()
        
        # Passaggio in avanti
        output = self.model(input_image)
        
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        # Passaggio all'indietro per la classe target
        self.model.zero_grad()
        output[0, class_idx].backward()
        
        # Verifica che abbiamo catturato gradienti e attivazioni
        if self.gradients is None or self.activations is None:
            raise ValueError("Hooks non hanno catturato gradienti/attivazioni!")
        
        # Calcola GradCAM
        # Media dei gradienti sulle dimensioni spaziali (larghezza, altezza)
        gradients = self.gradients[0]  # [C, H, W]
        activations = self.activations[0]  # [C, H, W]
        
        # Pesi: media dei gradienti per ogni canale
        weights = gradients.mean(dim=(1, 2), keepdim=True)  # [C, 1, 1]
        
        # Combinazione pesata delle attivazioni
        cam = (weights * activations).sum(dim=0)  # [H, W]
        
        # Normalizza tra 0 e 1
        cam = torch.relu(cam)  # Solo valori positivi
        cam = cam / (cam.max() + 1e-8)  # Normalizza
        
        return cam.detach().cpu().numpy(), class_idx

# Scegli il layer target: l'ultimo layer convoluzionale di ResNet18
# model.layer4[-1] è l'ultimo blocco residuo, che contiene le feature più semantiche
target_layer = model.layer4[-1].conv2
print(f"Layer target per GradCAM: {target_layer}")

# Crea GradCAM
gradcam = GradCAM(model, target_layer)

In [ ]:
# Funzione per visualizzare GradCAM (solo matplotlib, no cv2)
def visualize_gradcam(image_path, model, gradcam, transform, class_names, device):
    """Visualizza immagine originale con overlay GradCAM"""
    from PIL import Image
    
    # Carica immagine originale
    original_img = Image.open(image_path).convert('RGB')
    original_img_np = np.array(original_img)
    
    # Trasforma per il modello
    img_tensor = transform(original_img).unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    
    # Genera CAM
    cam, pred_class_idx = gradcam.generate_cam(img_tensor)
    pred_class = class_names[pred_class_idx]
    
    # Resize CAM all'immagine originale usando PIL
    from PIL import Image as PILImage
    cam_pil = PILImage.fromarray((cam * 255).astype(np.uint8))
    cam_resized = cam_pil.resize((original_img_np.shape[1], original_img_np.shape[0]), PILImage.BILINEAR)
    cam_resized = np.array(cam_resized) / 255.0
    
    # Applica colormap usando matplotlib (jet: blu=low, rosso=high)
    import matplotlib.cm as cm
    colormap = cm.get_cmap('jet')
    cam_colored = colormap(cam_resized)[:, :, :3]  # Rimuovi alpha channel
    cam_colored = (cam_colored * 255).astype(np.uint8)
    
    # Overlay: combina immagine originale e heatmap
    overlay = (0.6 * original_img_np + 0.4 * cam_colored).astype(np.uint8)
    
    return original_img_np, overlay, pred_class, pred_class_idx

# Trova un esempio corretto e uno sbagliato
correct_examples = []
wrong_examples = []

for idx in range(len(test_dataset)):
    image_path, true_label_idx = test_dataset.samples[idx]
    img_tensor = val_test_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        pred_idx = output.argmax(dim=1).item()
    
    if pred_idx == true_label_idx and len(correct_examples) < 1:
        correct_examples.append(idx)
    elif pred_idx != true_label_idx and len(wrong_examples) < 1:
        wrong_examples.append(idx)
    
    if len(correct_examples) >= 1 and len(wrong_examples) >= 1:
        break

# Visualizza esempi
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

from PIL import Image

examples_to_show = []
if correct_examples:
    examples_to_show.append(("Corretto", correct_examples[0]))
if wrong_examples:
    examples_to_show.append(("Sbagliato", wrong_examples[0]))

for row, (title, idx) in enumerate(examples_to_show):
    image_path, true_label_idx = test_dataset.samples[idx]
    true_label = class_names[true_label_idx]
    
    original, overlay, pred_label, _ = visualize_gradcam(
        image_path, model, gradcam, val_test_transform, class_names, device
    )
    
    # Immagine originale
    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f"{title} - Originale\nTrue: {true_label}, Pred: {pred_label}")
    axes[row, 0].axis('off')
    
    # Con GradCAM overlay
    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title(f"{title} - GradCAM\nZone rosse = più importanti")
    axes[row, 1].axis('off')

# Nascondi assi non usati
if len(examples_to_show) < 2:
    axes[1, 0].axis('off')
    axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\nSpiegazione GradCAM:")
print("- Le zone ROSSE/GIALLE indicano dove la rete 'guarda' per fare la predizione")
print("- Le zone BLU/SCURE sono meno importanti per la decisione")
print("- Una buona rete dovrebbe concentrarsi sulle parti rilevanti del fiore (petali, centro, etc.)")